In [1]:
pip install torch transformers scikit-learn pandas wandb

Note: you may need to restart the kernel to use updated packages.


In [2]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Thimathi\_netrc.
wandb: Currently logged in as: chathuradissanayake274 (chathuradissanayake274-chathura) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
pip install wandb

Note: you may need to restart the kernel to use updated packages.


In [4]:
import wandb

info = wandb.login()
print(info)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Thimathi\_netrc.
wandb: Currently logged in as: chathuradissanayake274 (chathuradissanayake274-chathura) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True


In [5]:
import random
import os
import wandb

# -----------------------
# AUTO LOGIN
# -----------------------
wandb.login(key="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")

# -----------------------
# START RUN
# -----------------------
run = wandb.init(
    entity="chathuradissanayake274-chathura",
    project="2nd",
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# -----------------------
# SIMULATED TRAINING
# -----------------------
epochs = 10
offset = random.random() / 5

for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    run.log({
        "epoch": epoch,
        "accuracy": acc,
        "loss": loss
    })

run.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Thimathi\_netrc


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁▇███▇██
epoch,▁▂▃▄▅▆▇█
loss,█▆▃▆▄▁▂▁
accuracy,0.80474
epoch,9
loss,0.16046


In [6]:
import torch
import pandas as pd
import numpy as np
import wandb
import accelerate
print(accelerate.__version__)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer


# -----------------------
# W&B INIT
# -----------------------
wandb.init(project="Aurevia_Burnout_Detection_Text", name="bert_binary_v1")

# -----------------------
# LOAD DATA
# -----------------------

import pandas as pd

df = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_1.csv")
df2 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_2.csv")
df3 = pd.read_csv(r"C:\Users\Thimathi\source\Aurevia\data\raw\GoEmotions\goemotions_3.csv")

df = pd.concat([df, df2, df3], ignore_index=True)

# -----------------------
# DEFINE BURNOUT EMOTIONS
# -----------------------
burnout_emotions = [
    'anger', 'annoyance', 'disappointment',
    'disapproval', 'disgust', 'fear',
    'grief', 'nervousness', 'remorse',
    'sadness'
]

def convert_to_binary(row):
    for emotion in burnout_emotions:
        if row[emotion] == 1:
            return 1
    return 0

df["binary_label"] = df.apply(convert_to_binary, axis=1)

# -----------------------
# BALANCE DATASET (Downsample)
# -----------------------
majority = df[df.binary_label == 0]
minority = df[df.binary_label == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

sample_size = 20000

majority_sample = majority.sample(n=sample_size, random_state=42)
minority_sample = minority.sample(n=sample_size, random_state=42)

df_balanced = pd.concat([majority_sample, minority_sample])

# -----------------------
# TRAIN TEST SPLIT
# -----------------------
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced["text"],
    df_balanced["binary_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["binary_label"]
)

# -----------------------
# TOKENIZER
# -----------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

class BurnoutDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BurnoutDataset(train_encodings, train_labels)
val_dataset = BurnoutDataset(val_encodings, val_labels)

# -----------------------
# MODEL
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# -----------------------
# METRICS
# -----------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# -----------------------
# TRAINING CONFIG
# -----------------------
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # ← IMPORTANT
    save_strategy="epoch",            # ← MUST MATCH
    logging_strategy="steps",
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="wandb",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model("./aurevia_text_model")

wandb.finish()

c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.12.0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 226.15it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.485757,0.465851,0.776250,0.794819,0.733912,0.866750
2,0.404932,0.472371,0.778625,0.787267,0.757688,0.819250


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]
c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'ber

eval/accuracy,▁█
eval/f1,█▁
eval/loss,▁█
eval/precision,▁█
eval/recall,█▁
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
train/epoch,▁▂▃▃▄▄▅▆▆▇███
train/global_step,▁▂▃▃▄▄▅▆▆▇███
+3,...
